# Checkpoint charts
Two charts for the checkpoint meeting. Uses the cleaned files from `01_data_prep.ipynb`.

In [1]:
import pandas as pd
import altair as alt

hits = pd.read_csv("data/processed/hits_clean.csv")
all_music = pd.read_csv("data/processed/all_music_yearly.csv")
genres = pd.read_csv("data/processed/genre_family_yearly.csv")

In [2]:
# only use hits that matched to spotify, and stop at 2022 since 2023 is only half a year
hits = hits[(hits["matched"] == True) & (hits["debut_year"] <= 2022)]
all_music = all_music[all_music["year"] <= 2022]
genres = genres[genres["year"] <= 2022]

hits.shape

(7657, 32)

## Chart 1: median song length by year

In [3]:
hits_by_year = hits.groupby("debut_year")["duration_min"].median().reset_index()
hits_by_year.columns = ["year", "median_length"]
hits_by_year["group"] = "Billboard Hot 100 hits"

all_by_year = all_music[["year", "median"]].copy()
all_by_year.columns = ["year", "median_length"]
all_by_year["group"] = "All Spotify songs"

lengths = pd.concat([hits_by_year, all_by_year])
lengths.head()

,year,median_length,group
0,2000,4.071550,Billboard Hot 100 hits
1,2001,4.033333,Billboard Hot 100 hits
2,2002,4.010667,Billboard Hot 100 hits
3,2003,4.010000,Billboard Hot 100 hits
4,2004,3.936883,Billboard Hot 100 hits


In [4]:
chart1 = alt.Chart(lengths).mark_line(point=True).encode(
    x=alt.X("year:O", title="Year"),
    y=alt.Y("median_length:Q", title="Median song length (minutes)", scale=alt.Scale(zero=False)),
    color=alt.Color("group:N", title="")
).properties(
    title="Median Song Length by Year (2000-2022)",
    width=600,
    height=350
)

chart1

alt.Chart(...)

Hits went from about 4:04 in 2000 to about 3:12 in 2022. All songs on Spotify stay around 3:55 until about 2015 and then start dropping too.

## Chart 2: median length by genre and year

In [5]:
chart2 = alt.Chart(genres).mark_rect().encode(
    x=alt.X("year:O", title="Year"),
    y=alt.Y("genre_family:N", title="Genre"),
    color=alt.Color("median:Q", title="Median length (min)", scale=alt.Scale(scheme="blues"))
).properties(
    title="Median Song Length by Genre and Year (all Spotify songs)",
    width=600,
    height=250
)

chart2

alt.Chart(...)

Every genre gets lighter (shorter) toward the right side, mostly after about 2015. Electronic/dance and hip-hop drop the most, and rock barely changes.

In [6]:
# save pngs for the meeting
chart1.save("figures/simple_line_chart.png")
chart2.save("figures/simple_heatmap.png")